# From genes to prognosis: predicting breast-cancer mortality from clinical and genomic data (METABRIC)

**Machine Learning 1 capstone, Tek-Up, ING-4 SDIA, 2026–2027**
**Team:** Oumaima Hafiane, Imen Hichri

# 1. Problem Statement

## Problem description
Breast cancer is the most commonly diagnosed cancer in women worldwide. After diagnosis, oncologists estimate each patient's prognosis to decide how aggressive the treatment should be. A standard tool is the Nottingham Prognostic Index (NPI), a simple formula based on three clinical measurements: tumor size, number of positive lymph nodes and tumor grade. The METABRIC study adds molecular data for 1,904 patients: the expression of 489 genes and the mutation status of 173 genes. Our question is: **do gene expression and mutations predict death from breast cancer better than clinical data alone, and better than the NPI that oncologists already use?**

## Task type
Supervised **binary classification**.

- **Target:** `y = 1` if `death_from_cancer == "Died of Disease"` (622 patients, 33%), `y = 0` otherwise ("Living" or "Died of Other Causes").
- **Features:** 27 clinical variables (age, tumor size, grade, receptor status, treatments, ...), 489 gene-expression z-scores and 173 gene-mutation indicators.
- **Excluded to avoid leakage:** `overall_survival`, `overall_survival_months`, `death_from_cancer`, `patient_id`.

## Primary metric: ROC-AUC
- **Accuracy is misleading here:** a model that predicts "no death" for everyone already reaches 67% accuracy while detecting no patient at risk.
- **ROC-AUC measures ranking quality:** it is the probability that a patient who died of the disease receives a higher risk score than a patient who did not. It does not depend on a decision threshold.
- **It allows a fair comparison with the NPI:** the NPI is a score, not a yes/no prediction, so it can be evaluated with ROC-AUC exactly like our models.

**Secondary metrics:** recall, precision and F1 for the positive class, and the precision–recall curve. Recall matters most among them, because missing a high-risk patient (false negative) is more costly than a false alarm.

## Why it matters and who would use it
A better prognosis estimate helps avoid under-treating high-risk patients and over-treating low-risk ones. The end users would be oncologists and cancer researchers. Measuring gene expression is expensive, so it is also useful to know whether it adds predictive value beyond cheap clinical measurements. This project is an academic study, not a clinical tool.

## Known limitation
Patients were followed for different lengths of time, so some "Living" patients may have died of the disease after the study ended. The proper tool for this is survival analysis, which is outside the scope of this course. We treat the problem as binary classification and discuss this limitation in the conclusion.

# 2. Data Loading & Dictionary

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
pd.set_option('display.max_colwidth', None)

In [2]:
DATA_PATH = '../data/raw/METABRIC_RNA_Mutation.csv'

# low_memory=False: pandas reads each column in one pass, so mutation columns
# that mix "0" and text codes get one consistent type instead of a warning
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape: {df.shape}")
print(f"Duplicated rows: {df.duplicated().sum()}")
print(f"Duplicated patient IDs: {df['patient_id'].duplicated().sum()}")
df.head()

Shape: (1904, 693)
Duplicated rows: 0
Duplicated patient IDs: 0


,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,...,mtap_mut,ppp2cb_mut,smarcd1_mut,nras_mut,ndfip1_mut,hras_mut,prps2_mut,smarcb1_mut,stmn2_mut,siah1_mut
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0


In [3]:
clinical_cols = df.columns[:31].tolist()
mut_cols = [c for c in df.columns if c.endswith('_mut')]
expr_cols = [c for c in df.columns if c not in clinical_cols and c not in mut_cols]

print(f"Clinical: {len(clinical_cols)} | Gene expression: {len(expr_cols)} | Mutations: {len(mut_cols)}")
assert (len(clinical_cols), len(expr_cols), len(mut_cols)) == (31, 489, 173)

Clinical: 31 | Gene expression: 489 | Mutations: 173


In [4]:
df.info()                    # 693 columns: pandas prints a summary only
df[clinical_cols].info()     # full detail for the clinical columns

<class 'pandas.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Columns: 693 entries, patient_id to siah1_mut
dtypes: float64(498), int64(5), str(190)
memory usage: 10.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Data columns (total 31 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   patient_id                      1904 non-null   int64  
 1   age_at_diagnosis                1904 non-null   float64
 2   type_of_breast_surgery          1882 non-null   str    
 3   cancer_type                     1904 non-null   str    
 4   cancer_type_detailed            1889 non-null   str    
 5   cellularity                     1850 non-null   str    
 6   chemotherapy                    1904 non-null   int64  
 7   pam50_+_claudin-low_subtype     1904 non-null   str    
 8   cohort                          1904 non-null   float64
 9   er_status_measured_by_ihc       1874 non-null   str    


In [5]:
# Clinical, numerical
display(df[clinical_cols].describe().T.round(2))

# Clinical, categorical
display(df[clinical_cols].describe(exclude='number').T)

# Gene expression: one row per gene, then summarised across the 489 genes
expr_stats = df[expr_cols].agg(['mean', 'std', 'min', 'max']).T
display(expr_stats.describe().round(2))

# Mutations: share of cells that are not "0"
mutated = df[mut_cols].astype(str) != '0'
print(f"Mutated cells: {mutated.mean().mean():.1%}")
print(f"Missing values outside clinical columns: {df[expr_cols + mut_cols].isna().sum().sum()}")

,count,mean,std,min,25%,50%,75%,max
patient_id,1904.0,3921.98,2358.48,0.00,896.50,4730.50,5536.25,7299.00
age_at_diagnosis,1904.0,61.09,12.98,21.93,51.38,61.77,70.59,96.29
chemotherapy,1904.0,0.21,0.41,0.00,0.00,0.00,0.00,1.00
cohort,1904.0,2.64,1.23,1.00,1.00,3.00,3.00,5.00
neoplasm_histologic_grade,1832.0,2.42,0.65,1.00,2.00,3.00,3.00,3.00
hormone_therapy,1904.0,0.62,0.49,0.00,0.00,1.00,1.00,1.00
lymph_nodes_examined_positive,1904.0,2.00,4.08,0.00,0.00,0.00,2.00,45.00
mutation_count,1859.0,5.70,4.06,1.00,3.00,5.00,7.00,80.00
nottingham_prognostic_index,1904.0,4.03,1.14,1.00,3.05,4.04,5.04,6.36
overall_survival_months,1904.0,125.12,76.33,0.00,60.83,115.62,184.72,355.20


,count,unique,top,freq
type_of_breast_surgery,1882,2,MASTECTOMY,1127
cancer_type,1904,2,Breast Cancer,1903
cancer_type_detailed,1889,6,Breast Invasive Ductal Carcinoma,1500
cellularity,1850,3,High,939
pam50_+_claudin-low_subtype,1904,7,LumA,679
er_status_measured_by_ihc,1874,2,Positve,1445
er_status,1904,2,Positive,1459
her2_status_measured_by_snp6,1904,4,NEUTRAL,1383
her2_status,1904,2,Negative,1668
tumor_other_histologic_subtype,1889,8,Ductal/NST,1454


,mean,std,min,max
count,489.0,489.0,489.00,489.00
mean,-0.0,1.0,-3.05,5.80
std,0.0,0.0,1.04,2.75
min,-0.0,1.0,-7.24,2.00
25%,-0.0,1.0,-3.62,4.15
50%,-0.0,1.0,-2.96,5.13
75%,0.0,1.0,-2.31,6.50
max,0.0,1.0,-0.92,20.40


Mutated cells: 3.0%
Missing values outside clinical columns: 0


In [6]:
clinical_dict = [
    # (feature, type, description, unit)
    ('patient_id', 'Identifier', 'Patient identifier (not a feature)', '-'),
    ('age_at_diagnosis', 'Numerical, continuous', 'Age of the patient at diagnosis', 'years'),
    ('type_of_breast_surgery', 'Categorical, binary', 'Mastectomy or breast-conserving surgery', '-'),
    ('cancer_type', 'Categorical, nominal', 'Breast cancer or breast sarcoma', '-'),
    ('cancer_type_detailed', 'Categorical, nominal', 'Detailed histological type (ductal, lobular, mixed, ...)', '-'),
    ('cellularity', 'Categorical, ordinal', 'Proportion of tumor cells in the sample: Low < Moderate < High', '-'),
    ('chemotherapy', 'Categorical, binary', 'Patient received chemotherapy (1) or not (0)', '-'),
    ('pam50_+_claudin-low_subtype', 'Categorical, nominal', 'Molecular subtype from the PAM50 gene signature (LumA, LumB, Her2, Basal, ...)', '-'),
    ('cohort', 'Categorical, nominal', 'Group of patients the sample belongs to in the study (1 to 5)', '-'),
    ('er_status_measured_by_ihc', 'Categorical, binary', 'Estrogen-receptor status measured by immunohistochemistry', '-'),
    ('er_status', 'Categorical, binary', 'Estrogen-receptor status (Positive / Negative)', '-'),
    ('neoplasm_histologic_grade', 'Categorical, ordinal', 'How abnormal the tumor cells look: 1 (low) to 3 (high)', '-'),
    ('her2_status_measured_by_snp6', 'Categorical, nominal', 'HER2 gene copy number measured by SNP6 array (neutral, gain, loss)', '-'),
    ('her2_status', 'Categorical, binary', 'HER2 receptor status (Positive / Negative)', '-'),
    ('tumor_other_histologic_subtype', 'Categorical, nominal', 'Histological subtype (Ductal/NST, Lobular, Mixed, ...)', '-'),
    ('hormone_therapy', 'Categorical, binary', 'Patient received hormone therapy (1) or not (0)', '-'),
    ('inferred_menopausal_state', 'Categorical, binary', 'Pre- or post-menopausal, inferred from age', '-'),
    ('integrative_cluster', 'Categorical, nominal', 'Molecular subgroup (IntClust) defined by Curtis et al. 2012', '-'),
    ('primary_tumor_laterality', 'Categorical, binary', 'Breast affected: Left or Right', '-'),
    ('lymph_nodes_examined_positive', 'Numerical, discrete', 'Number of lymph nodes containing cancer cells', 'count'),
    ('mutation_count', 'Numerical, discrete', 'Number of mutated genes found in the tumor', 'count'),
    ('nottingham_prognostic_index', 'Numerical, continuous', 'NPI = 0.2 x size (cm) + lymph-node stage + grade; higher = worse prognosis', 'score'),
    ('oncotree_code', 'Categorical, nominal', 'OncoTree code of the cancer type (IDC, ILC, MDLC, ...)', '-'),
    ('overall_survival_months', 'Numerical, continuous', 'Time from diagnosis to death or last follow-up (LEAKAGE, removed)', 'months'),
    ('overall_survival', 'Categorical, binary', 'Alive (1) or dead (0) at last follow-up (LEAKAGE, removed)', '-'),
    ('pr_status', 'Categorical, binary', 'Progesterone-receptor status (Positive / Negative)', '-'),
    ('radio_therapy', 'Categorical, binary', 'Patient received radiotherapy (1) or not (0)', '-'),
    ('3-gene_classifier_subtype', 'Categorical, nominal', 'Subtype from a 3-gene classifier (ER, HER2, proliferation)', '-'),
    ('tumor_size', 'Numerical, continuous', 'Diameter of the tumor', 'mm'),
    ('tumor_stage', 'Categorical, ordinal', 'Stage of the cancer: 0 (early) to 4 (advanced)', '-'),
    ('death_from_cancer', 'Categorical, nominal', 'Living / Died of Disease / Died of Other Causes (source of the TARGET)', '-'),
]

data_dict = pd.DataFrame(clinical_dict, columns=['Feature', 'Type', 'Description', 'Unit'])
assert data_dict['Feature'].tolist() == clinical_cols   # every clinical column is documented

data_dict['Dtype'] = data_dict['Feature'].map(df.dtypes.astype(str))
data_dict['Unique values'] = data_dict['Feature'].map(df[clinical_cols].nunique())
data_dict['Missing %'] = data_dict['Feature'].map((df[clinical_cols].isna().mean() * 100).round(1))

genomic_rows = pd.DataFrame([
    {'Feature': f'{len(expr_cols)} gene-expression columns (brca1, ...)', 'Type': 'Numerical, continuous',
     'Description': 'mRNA expression level of one gene, standardised across patients', 'Unit': 'z-score',
     'Dtype': 'float64', 'Unique values': '-', 'Missing %': 0.0},
    {'Feature': f'{len(mut_cols)} mutation columns (*_mut)', 'Type': 'Text, binarised later',
     'Description': '"0" = gene not mutated; otherwise the mutation code (e.g. H1047R)', 'Unit': '-',
     'Dtype': 'object', 'Unique values': '-', 'Missing %': 0.0},
])

data_dict = pd.concat([data_dict, genomic_rows], ignore_index=True)
data_dict

,Feature,Type,Description,Unit,Dtype,Unique values,Missing %
0,patient_id,Identifier,Patient identifier (not a feature),-,int64,1904,0.0
1,age_at_diagnosis,"Numerical, continuous",Age of the patient at diagnosis,years,float64,1572,0.0
2,type_of_breast_surgery,"Categorical, binary",Mastectomy or breast-conserving surgery,-,str,2,1.2
3,cancer_type,"Categorical, nominal",Breast cancer or breast sarcoma,-,str,2,0.0
4,cancer_type_detailed,"Categorical, nominal","Detailed histological type (ductal, lobular, mixed, ...)",-,str,6,0.8
5,cellularity,"Categorical, ordinal",Proportion of tumor cells in the sample: Low < Moderate < High,-,str,3,2.8
6,chemotherapy,"Categorical, binary",Patient received chemotherapy (1) or not (0),-,int64,2,0.0
7,pam50_+_claudin-low_subtype,"Categorical, nominal","Molecular subtype from the PAM50 gene signature (LumA, LumB, Her2, Basal, ...)",-,str,7,0.0
8,cohort,"Categorical, nominal",Group of patients the sample belongs to in the study (1 to 5),-,float64,5,0.0
9,er_status_measured_by_ihc,"Categorical, binary",Estrogen-receptor status measured by immunohistochemistry,-,str,2,1.6


**Summary.** The dataset has 1,904 patients and 693 columns, with no duplicated rows or patients. The columns form three groups: 31 clinical columns (numerical and categorical), 489 gene-expression z-scores and 173 mutation columns stored as text. Missing values appear only in the clinical columns. Four clinical columns will not be used as features: `patient_id` (identifier), and `overall_survival`, `overall_survival_months` and `death_from_cancer`, which are known only after the outcome.